# FaithfulMed Extractor Agent

This notebook is the working prototype for the Extractor agent. It loads MTSamples notes, splits the transcription into clinical sections, and extracts structured facts that later agents can use.

The goal is to convert raw transcribed medical text into a clean list of atomic facts such as diagnoses, medications, allergies, procedures, symptoms, and lab values.

In [1]:
# Set the path to the MTSamples CSV used for this extractor prototype.
# This lets the notebook find the data folder without depending on a specific local machine path.
from pathlib import Path
import csv
import json
import re

DATA_PATH = Path.cwd().parent / "data" / "mtsamples - mtsamples.csv"
print(DATA_PATH.exists())
print(DATA_PATH)

True
c:\Users\Samuel\pythonprojects\Tracking_software\Google-2E-faithfulmed-a-multi-agent-system-for-faithful-patient-friendly-medical-explanations\data\mtsamples - mtsamples.csv


In [20]:
# This function cleans raw transcription text while preserving clinical punctuation.
def clean_text(text: str) -> str:
    value = re.sub(r"(?<=[.!?])(?=[A-Z])", " ", text or "")
    value = re.sub(r"(?<=[.!?])(?=\d+\.)", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    value = re.sub(r"^[\s,;:.-]+|[\s,;:.-]+$", "", value)
    return value


# This helper removes formatting artifacts from extracted values without discarding internal punctuation.
def normalize_fact_value(text: str) -> str:
    value = clean_text(text)
    value = re.sub(r"\s+([,.;:])", r"\1", value)
    value = re.sub(r"([,.;:]){2,}", ".", value)
    value = re.sub(r"\s+\.", ".", value)
    value = re.sub(r"\.+$", "", value)
    value = re.sub(r"^\.+", "", value)
    return clean_text(value)


# Recognized labels avoid treating capitalized words in ordinary prose as section headers.
SECTION_LABELS = (
    "PREOPERATIVE DIAGNOSES", "POSTOPERATIVE DIAGNOSES", "PREOPERATIVE DIAGNOSIS",
    "POSTOPERATIVE DIAGNOSIS", "POST OPERATIVE DIAGNOSES", "POST OPERATIVE DIAGNOSIS",
    "PROCEDURES PERFORMED", "PROCEDURE PERFORMED", "OPERATION PERFORMED",
    "PROCEDURE IN DETAIL", "PROCEDURES", "PROCEDURE", "OPERATION",
    "HISTORY OF PRESENT ILLNESS", "REVIEW OF SYSTEMS", "PAST MEDICAL HISTORY",
    "PHYSICAL EXAMINATION", "PHYSICAL EXAM", "REASON FOR EXAM", "INITIAL STUDIES",
    "ESTIMATED BLOOD LOSS", "ALLERGIES TO MEDICATIONS", "CURRENT MEDICATIONS",
    "HOME MEDICATIONS", "MEDICATIONS", "ALLERGIES", "CHIEF COMPLAINT",
    "ASSESSMENT AND PLAN", "ASSESSMENT/PLAN", "ASSESSMENT", "IMPRESSION",
    "DIAGNOSES", "DIAGNOSIS", "PRESENT ILLNESS", "MEDICAL HISTORY", "VITAL SIGNS",
    "INDICATIONS", "ANESTHESIA", "COMPLICATIONS", "DISPOSITION", "FINDINGS",
    "TECHNIQUE", "CONCLUSION", "RECOMMENDATION", "SUMMARY", "COURSE", "HISTORY",
    "DOPPLER", "EXAMINATION", "OBJECTIVE", "SUBJECTIVE", "VITALS", "HEENT",
    "CARDIOVASCULAR", "ABDOMEN", "EXTREMITIES", "LUNGS", "NECK", "EXAM", "PLAN",
    "LABS", "2-D M-MODE", "PMH", "FHX", "SHX", "ROS", "HX", "CC", "CN",
)


# Split notes on known clinical headings and their common comma-separated CSV formatting.
def get_sections(transcription: str) -> dict:
    if not transcription:
        return {}

    labels = "|".join(re.escape(label) for label in sorted(SECTION_LABELS, key=len, reverse=True))
    pattern = re.compile(
        rf"(?:^|(?<=[.!?,])|(?<=\n))\s*(?P<label>{labels})\s*[:.,]+(?=\s|[A-Z0-9(])",
        flags=re.IGNORECASE,
    )
    matches = list(pattern.finditer(transcription))
    sections = {}

    for idx, match in enumerate(matches):
        label = match.group("label").upper()
        start = match.end()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(transcription)
        value = clean_text(transcription[start:end])
        if value:
            sections[label] = clean_text(f"{sections[label]} {value}") if label in sections else value

    if not sections:
        sections["FULL_TEXT"] = clean_text(transcription)

    return sections


# Extract structured facts for downstream agents.
def extract_clinical_atoms(transcription: str) -> list[dict]:
    sections = get_sections(transcription)
    atoms = []

    def add_atom(atom_type: str, value: str, source: str = "", confidence: float = 1.0) -> None:
        if not value:
            return
        normalized_value = normalize_fact_value(value)
        if not normalized_value:
            return
        atoms.append({
            "type": atom_type,
            "value": normalized_value,
            "source": clean_text(source),
            "confidence": confidence,
        })

    for key in [
        "ASSESSMENT", "ASSESSMENT/PLAN", "IMPRESSION", "DIAGNOSIS", "DIAGNOSES",
        "POSTOPERATIVE DIAGNOSIS", "POSTOPERATIVE DIAGNOSES",
        "POST OPERATIVE DIAGNOSIS", "POST OPERATIVE DIAGNOSES", "PREOPERATIVE DIAGNOSIS",
        "PREOPERATIVE DIAGNOSES",
    ]:
        if key in sections:
            add_atom("diagnosis", sections[key], key, confidence=0.95)

    if "ASSESSMENT AND PLAN" in sections:
        add_atom("clinical_assessment", sections["ASSESSMENT AND PLAN"], "ASSESSMENT AND PLAN", confidence=0.88)
    if "PLAN" in sections:
        add_atom("treatment_plan", sections["PLAN"], "PLAN", confidence=0.88)

    for key in ["MEDICATIONS", "CURRENT MEDICATIONS", "HOME MEDICATIONS"]:
        if key in sections:
            add_atom("medication", sections[key], key, confidence=0.92)

    for key in ["ALLERGIES", "ALLERGIES TO MEDICATIONS"]:
        if key in sections:
            add_atom("allergy", sections[key], key, confidence=0.93)

    for key in ["PROCEDURE", "PROCEDURE PERFORMED", "PROCEDURES PERFORMED", "OPERATION", "OPERATION PERFORMED", "PROCEDURES"]:
        if key in sections:
            add_atom("procedure", sections[key], key, confidence=0.9)

    for key in ["FINDINGS", "2-D M-MODE", "DOPPLER", "OBJECTIVE", "VITALS"]:
        if key in sections:
            add_atom("clinical_finding", sections[key], key, confidence=0.88)

    for key in ["CHIEF COMPLAINT", "HISTORY OF PRESENT ILLNESS", "REVIEW OF SYSTEMS", "SUBJECTIVE", "CC", "PMH", "ROS"]:
        if key in sections:
            text = sections[key]
            if len(text) < 300:
                add_atom("symptom_or_history", text, key, confidence=0.88)

    # Restrict numeric extraction to common measurements and laboratory fields.
    lab_pattern = re.compile(
        r"\b((?:BP|BLOOD PRESSURE|HR|HEART RATE|RR|RESPIRATORY RATE|T|TEMP|TEMPERATURE|"
        r"O2 SAT|OXYGEN SATURATION|WBC|HGB|HEMOGLOBIN|HCT|HEMATOCRIT|GLUCOSE|SODIUM|"
        r"POTASSIUM|CREATININE|BUN|PLATELETS|INR|ALT|AST)\s*[:=]?\s*"
        r"[0-9]+(?:\.[0-9]+)?(?:/[0-9]+(?:\.[0-9]+)?)?\s*"
        r"(?:mg/dL|mmHg|%|mEq/L|mmol/L|units|cm|kg|lbs|bpm|C|F|mm)?)\b",
        flags=re.I,
    )
    for field_name, value in sections.items():
        for match in lab_pattern.finditer(value):
            add_atom("lab_value", match.group(1), field_name, confidence=0.85)

    risk_pattern = re.compile(r"\b(smoker|diabetes|hypertension|asthma|obesity|stroke|heart disease|sleep apnea)\b", flags=re.I)
    for field_name, value in sections.items():
        for match in risk_pattern.finditer(value):
            add_atom("risk_factor", match.group(0), field_name, confidence=0.8)

    if not atoms:
        add_atom("raw_text", transcription, "FULL_TEXT", confidence=0.5)

    return atoms


# Load every MTSamples CSV row as a dictionary.
def load_mtsamples(path: Path = DATA_PATH) -> list[dict]:
    with open(path, newline="", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f)
        return list(reader)

In [22]:
# This demo block loads the first MTSamples row, extracts atoms from its transcription, and prints
# the output. It is meant to validate that the pipeline works on a real clinical example.
rows = load_mtsamples()
sample = rows[0]
transcription = sample.get("transcription", "")
print(f"Loaded {len(rows)} MTSamples records.")
print("Sample specialty:", sample.get("medical_specialty", "Unknown"))
print("Sample name:", sample.get("sample_name", "Unknown"))
print("\nExtracted atoms from the first note:\n")
print(json.dumps(extract_clinical_atoms(transcription)[:8], indent=2, ensure_ascii=False))

Loaded 4999 MTSamples records.
Sample specialty: Allergy / Immunology
Sample name: Allergic Rhinitis

Extracted atoms from the first note:

[
  {
    "type": "diagnosis",
    "value": "Allergic rhinitis",
    "source": "ASSESSMENT",
    "confidence": 0.95
  },
  {
    "type": "treatment_plan",
    "value": "1. She will try Zyrtec instead of Allegra again. Another option will be to use loratadine. She does not think she has prescription coverage so that might be cheaper. 2. Samples of Nasonex two sprays in each nostril given for three weeks. A prescription was written as well",
    "source": "PLAN",
    "confidence": 0.88
  },
  {
    "type": "medication",
    "value": "Her only medication currently is Ortho Tri-Cyclen and the Allegra",
    "source": "MEDICATIONS",
    "confidence": 0.92
  },
  {
    "type": "allergy",
    "value": "She has no known medicine allergies",
    "source": "ALLERGIES",
    "confidence": 0.93
  },
  {
    "type": "clinical_finding",
    "value": "Weight was 13

In [21]:
# Validate representative notes from different MTSamples specialties.
validation_expectations = {
    "Allergy / Immunology": {"SUBJECTIVE", "MEDICATIONS", "ALLERGIES", "VITALS", "ASSESSMENT", "PLAN"},
    "Surgery": {"PREOPERATIVE DIAGNOSES", "PROCEDURE PERFORMED"},
    "Cardiovascular / Pulmonary": {"2-D M-MODE", "DOPPLER"},
    "Neurology": {"CC", "HX", "PMH"},
    "Radiology": {"FINDINGS", "IMPRESSION"},
    "Obstetrics / Gynecology": {"POST OPERATIVE DIAGNOSES", "PROCEDURES"},
    "Gastroenterology": {"HISTORY OF PRESENT ILLNESS", "ASSESSMENT AND PLAN"},
}
expected_atom_keys = {"type", "value", "source", "confidence"}

for specialty, expected_sections in validation_expectations.items():
    sample = next(
        row for row in rows
        if row.get("medical_specialty", "").strip() == specialty
        and row.get("transcription", "").strip()
    )
    sections = get_sections(sample["transcription"])
    atoms = extract_clinical_atoms(sample["transcription"])
    assert expected_sections.issubset(sections), (specialty, expected_sections - set(sections))
    assert atoms and all(set(atom) == expected_atom_keys for atom in atoms), specialty
    print(f"\n{specialty}: {sample.get('sample_name', 'Unknown')}")
    print(f"Sections ({len(sections)}): {list(sections)}")
    print(f"Atom schema: consistent; types: {[atom['type'] for atom in atoms]}")
    for atom in atoms[:4]:
        print(f"  {atom['source']} -> {atom['value'][:140]}")


Allergy / Immunology: Allergic Rhinitis
Sections (9): ['SUBJECTIVE', 'MEDICATIONS', 'ALLERGIES', 'VITALS', 'HEENT', 'NECK', 'LUNGS', 'ASSESSMENT', 'PLAN']
Atom schema: consistent; types: ['diagnosis', 'treatment_plan', 'medication', 'allergy', 'clinical_finding', 'lab_value', 'risk_factor']
  ASSESSMENT -> Allergic rhinitis
  PLAN -> 1. She will try Zyrtec instead of Allegra again. Another option will be to use loratadine. She does not think she has prescription coverage 
  MEDICATIONS -> Her only medication currently is Ortho Tri-Cyclen and the Allegra
  ALLERGIES -> She has no known medicine allergies

Surgery: Youngswick Bunionectomy
Sections (5): ['PREOPERATIVE DIAGNOSES', 'POSTOPERATIVE DIAGNOSES', 'PROCEDURE PERFORMED', 'HISTORY', 'PROCEDURE IN DETAIL']
Atom schema: consistent; types: ['diagnosis', 'diagnosis', 'procedure']
  POSTOPERATIVE DIAGNOSES -> 1. Hallux rigidus, left foot. 2. Elevated first metatarsal, left foot
  PREOPERATIVE DIAGNOSES -> 1. Hallux rigidus, left foot. 